# RetinaX Notebook 01: Data Governance, Quality Assessment & Preprocessing Lab

> **Clinical Disclaimer:** RetinaX is an academic research prototype and must not be used for clinical diagnosis.
> When local Kaggle APTOS/EyePACS cohorts are unavailable, this notebook demonstrates the full pipeline on **clearly labelled synthetic phantom cohorts** (`is_synthetic=True`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import load_config, get_hardware_summary
from src.data.ingestion import discover_datasets, load_raw_records
from src.data.manifests import build_manifest, save_manifest
from src.data.splitting import assign_splits
from src.data.synthetic import generate_synthetic_demo_dataset
from src.preprocessing.quality_assessment import assess_fundus_quality
from src.preprocessing.fundus_processing import preprocess_fundus_image, load_rgb_array

print("Hardware summary:", get_hardware_summary())
cfg = load_config(preset_path=str(ROOT / "configs/lightweight_cpu.yaml"))

In [ ]:
status = discover_datasets()
if status["aptos"]["available"]:
    dataset_key = "aptos"
else:
    generate_synthetic_demo_dataset(num_primary=80, num_external=40, image_size=128, seed=42)
    dataset_key = "synthetic_primary"

raw_df = load_raw_records(dataset_key, seed=42)
manifest_df, summary = build_manifest(
    raw_df,
    processing_config=cfg["preprocessing"],
    quality_fn=assess_fundus_quality,
)
manifest_df, audit = assign_splits(manifest_df, seed=42)
print("Dataset:", dataset_key, "| Synthetic:", bool(manifest_df["is_synthetic"].any()))
print("Split audit:", audit)
manifest_df[["image_id", "original_label", "split", "quality_score", "quality_category", "is_synthetic"]].head(8)

In [ ]:
import matplotlib.pyplot as plt

sample_path = manifest_df.loc[~manifest_df["is_corrupt"], "abs_path"].iloc[0]
raw_rgb = load_rgb_array(sample_path)
presets = ["raw_resize", "fov_crop", "fov_clahe_norm", "ben_graham"]

fig, axes = plt.subplots(1, len(presets), figsize=(14, 3.5))
for ax, p in zip(axes, presets):
    ax.imshow(preprocess_fundus_image(raw_rgb, preset=p, image_size=128))
    ax.set_title(p)
    ax.axis("off")
plt.tight_layout()
plt.show()